In [142]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pyproj

torch.set_printoptions(sci_mode = False)

# Antarctica's hidden world - Antarctica's Gamburtsev Province Project (AGAP)

- Data from Bedmap2: https://ramadda.data.bas.ac.uk/repository/entry/show?entryid=synth%3A2fd95199-365e-4da1-ae26-3b6d48b3e6ac%3AL0JBU18yMDA3X0FHQVBfQUlSX0JNMi5jc3Y%3D
- Collected during polar year 2007 (rather 2008/2009)

- Load 2.5 M values
- skip first 18 rows in loading

Data:
- 66 flight trajectories
- Around 20 meter spacing between data points

In [143]:
bas_agab = pd.read_csv("~/data/bedmap/BAS_2007_AGAP_AIR_BM2.csv", skiprows = range(0, 18))
# on_bad_lines = "skip"

In [144]:
bas_agab

,trajectory_id,trace_number,longitude (degree_east),latitude (degree_north),date,time_UTC,surface_altitude (m),land_ice_thickness (m),bedrock_altitude (m),two_way_travel_time (m),aircraft_altitude (m),along_track_distance (m)
0,3,-9999,169.169538,-77.812387,2008-12-06,04:02:51.30,-21.97,-9999.0,-9999.0,-9999,-9999,-9999
1,3,-9999,169.170490,-77.812373,2008-12-06,04:02:51.63,-21.78,-9999.0,-9999.0,-9999,-9999,-9999
2,3,-9999,169.171442,-77.812358,2008-12-06,04:02:51.97,-21.56,-9999.0,-9999.0,-9999,-9999,-9999
3,3,-9999,169.172393,-77.812342,2008-12-06,04:02:52.30,-21.40,-9999.0,-9999.0,-9999,-9999,-9999
4,3,-9999,169.173341,-77.812326,2008-12-06,04:02:52.63,-21.24,-9999.0,-9999.0,-9999,-9999,-9999
...,...,...,...,...,...,...,...,...,...,...,...,...
2500530,69,-9999,163.941366,-79.305676,2009-01-16,06:46:59.48,3136.02,2180.1,955.9,-9999,-9999,-9999
2500531,69,-9999,163.941890,-79.305485,2009-01-16,06:46:59.81,3135.89,2180.0,955.9,-9999,-9999,-9999
2500532,69,-9999,163.942415,-79.305294,2009-01-16,06:47:00.15,3135.76,2179.6,956.1,-9999,-9999,-9999
2500533,69,-9999,163.942939,-79.305102,2009-01-16,06:47:00.48,3135.62,2180.3,955.3,-9999,-9999,-9999


In [145]:
bas_agab['trajectory_id'].value_counts() # 66
bas_agab['trace_number'].value_counts() # useless

bas_agab['date'].value_counts()
bas_agab['surface_altitude (m)'].value_counts()
bas_agab['bedrock_altitude (m)'].value_counts() # around 5000 values missing

bas_agab['two_way_travel_time (m)'].value_counts() # useless
bas_agab['aircraft_altitude (m)'].value_counts() # useless
bas_agab['along_track_distance (m)'].value_counts() # useless

along_track_distance (m)
-9999    2500535
Name: count, dtype: int64

In [146]:
# Remove use rows
bas_agab = bas_agab.drop(columns = ['trace_number', 
                                    'two_way_travel_time (m)',
                                    'aircraft_altitude (m)',
                                    'along_track_distance (m)',
                                    # Also drop date and time for now
                                    'date',
                                    'time_UTC'])

# Remove rows with no valid bedrock altitude value
bas_agab = bas_agab[bas_agab['bedrock_altitude (m)'] != - 9999.0]
# Remaining entries all have surface s, thickness h, and bed b 

In [147]:
bas_agab = bas_agab.rename(columns = {'land_ice_thickness (m)': 'h', 
                           'surface_altitude (m)': 's',
                           'bedrock_altitude (m)': 'b'
                           })

In [148]:
# This does not excatly add up
bas_agab["s"] - bas_agab["h"] - bas_agab["b"]

41819      0.10
41827     -0.05
41834      0.03
41847      0.25
41859     -0.07
           ... 
2500529   -0.04
2500530    0.02
2500531   -0.01
2500532    0.06
2500533    0.02
Length: 2030999, dtype: float64

In [149]:
bas_agab

,trajectory_id,longitude (degree_east),latitude (degree_north),s,h,b
41819,5,158.083968,-83.338681,1747.10,711.7,1035.3
41827,5,158.076561,-83.339409,1747.05,677.7,1069.4
41834,5,158.070221,-83.340067,1749.43,661.5,1087.9
41847,5,158.058873,-83.341330,1754.15,628.8,1125.1
41859,5,158.048821,-83.342529,1754.43,623.6,1130.9
...,...,...,...,...,...,...
2500529,69,163.940840,-79.305867,3136.16,2181.2,955.0
2500530,69,163.941366,-79.305676,3136.02,2180.1,955.9
2500531,69,163.941890,-79.305485,3135.89,2180.0,955.9
2500532,69,163.942415,-79.305294,3135.76,2179.6,956.1


# Convert to Polar Stereographic

In [150]:
# This is how projections work. check through tool
polarstereo_to_lonlat = pyproj.Transformer.from_crs(crs_from = pyproj.CRS("epsg:4326"),  
                                                    crs_to = pyproj.CRS("epsg:3031"),
                                                    always_xy = True) # xy order convention

# Pass in lon, lat and return x, y in polar stereo
x_array, y_array = polarstereo_to_lonlat.transform(
    bas_agab["longitude (degree_east)"], 
    bas_agab["latitude (degree_north)"])

bas_agab["x"] = x_array
bas_agab["y"] = y_array

# Find closest point / grid cell

# Find trajectory over Dome A (and associates flight line)

In [151]:
# max surface: Dome Argus
bas_agab[bas_agab["s"] == np.max(bas_agab["s"])]

,trajectory_id,longitude (degree_east),latitude (degree_north),s,h,b,x,y
320688,11,77.324434,-80.349617,4094.61,2224.0,1870.6,1.025310e+06,230604.281054
320689,11,77.324425,-80.349440,4094.61,2239.7,1854.9,1.025329e+06,230608.690941


In [152]:
# max bedrock:
bas_agab[bas_agab["b"] == np.max(bas_agab["b"])]

,trajectory_id,longitude (degree_east),latitude (degree_north),s,h,b,x,y
587088,18,79.365339,-79.127353,3764.41,810.6,2953.8,1.164405e+06,218641.572611


In [140]:
bas_agab[bas_agab["trajectory_id"] == 11]

,trajectory_id,longitude (degree_east),latitude (degree_north),s,h,b,x,y
297622,11,76.592038,-84.408402,3500.99,2853.5,647.4,5.914320e+05,140985.825570
297623,11,76.591230,-84.408240,3501.02,2838.7,662.3,5.914471e+05,140998.257202
297624,11,76.590419,-84.408078,3501.06,2840.3,660.7,5.914623e+05,141010.720260
297625,11,76.589607,-84.407916,3501.08,2839.2,661.8,5.914775e+05,141023.194099
297626,11,76.588795,-84.407754,3501.08,2846.9,654.1,5.914926e+05,141035.668396
...,...,...,...,...,...,...,...,...
336795,11,77.941237,-77.429358,3093.31,2433.4,659.9,1.340860e+06,286446.134438
336796,11,77.941255,-77.429181,3093.17,2437.0,656.2,1.340879e+06,286449.777716
336797,11,77.941272,-77.429005,3093.01,2427.8,665.2,1.340898e+06,286453.421422
336798,11,77.941288,-77.428829,3092.85,2426.5,666.4,1.340917e+06,286457.088523


In [154]:
bas_agab[bas_agab["trajectory_id"] == 18]

,trajectory_id,longitude (degree_east),latitude (degree_north),s,h,b,x,y
562001,18,74.443731,-78.437147,3338.83,2167.8,1171.0,1.214268e+06,338030.924337
562002,18,74.443235,-78.437332,3338.93,2179.6,1159.3,1.214246e+06,338035.992131
562003,18,74.442740,-78.437516,3339.03,2181.9,1157.2,1.214223e+06,338041.067797
562004,18,74.442244,-78.437700,3339.14,2183.8,1155.4,1.214201e+06,338046.164296
562005,18,74.441748,-78.437884,3339.25,2219.3,1120.0,1.214179e+06,338051.260434
...,...,...,...,...,...,...,...,...
606196,18,76.222766,-77.498447,3010.04,2765.8,244.3,1.324291e+06,324719.241821
606197,18,76.223077,-77.498295,3009.98,2768.9,241.0,1.324309e+06,324716.031852
606198,18,76.223387,-77.498144,3009.89,2763.3,246.5,1.324327e+06,324712.818639
606199,18,76.223698,-77.497991,3009.83,2760.3,249.6,1.324345e+06,324709.634473


In [158]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["y"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [160]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 900000]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 900000]["y"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

/tmp/ipykernel_329937/3797616491.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_329937/3797616491.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [199]:
flight_11_transect = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 1000000][bas_agab["x"] <= 1050000]
flight_11_transect_small = bas_agab[bas_agab["trajectory_id"] == 11][bas_agab["x"] >= 1032000][bas_agab["x"] <= 1035000]

/tmp/ipykernel_329937/2504367538.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_329937/2504367538.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_329937/2504367538.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_329937/2504367538.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [184]:
flight_11_transect.tail(n = 2)[["x", "y"]].astype(str)
# 4 meter y rise over 20 meters x
flight_11_transect.iloc[[400, 401, 402]].astype("str")

,x,y
321962,1049971.6161014307,234985.4034306404
321963,1049991.2857492368,234989.20897629694


In [172]:
fig = go.Figure(data = go.Scatter(x = flight_11_transect["x"], 
                                  y = flight_11_transect["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = flight_11_transect["x"], 
                                  y = flight_11_transect["s"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [205]:
fig = go.Figure(data = go.Scatter(x = flight_11_transect_small["x"], 
                                  y = flight_11_transect_small["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = flight_11_transect_small["x"], 
                                  y = flight_11_transect_small["s"], 
                                  mode = 'markers'))
fig.add_vline(x = 1032000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1032500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1033000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1033500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1034000, line_width = 1, line_color = "gray")
fig.add_vline(x = 1034500, line_width = 1, line_color = "gray")
fig.add_vline(x = 1035000, line_width = 1, line_color = "gray")
fig.update_layout(title = "Flight line 11 zoomed in with 500 m spacing", template = "plotly_white")
# Dome A is around 1 M
fig.show()

In [208]:
flight_11_transect_small["b"]

321036    2186.9
321037    2176.6
321038    2177.0
321039    2184.4
321040    2112.6
           ...  
321186    2284.0
321187    2291.2
321188    2292.4
321189    2299.7
321190    2299.8
Name: b, Length: 138, dtype: float64

In [210]:
# n = 138 
# std = 107.0, Normalized by N-0 by default
# var = 11,400.0
np.std(flight_11_transect_small["b"])

107.05814768831203

In [213]:
np.std(flight_11_transect_small["b"])

107.05814768831203

In [218]:
# Bed and height have an almost perfect negative correlation (complement definition)
np.corrcoef(flight_11_transect_small["b"], flight_11_transect_small["h"])

# Surprisingly the bed and  surface have a pretty strong negative relationship: Because surface height is capped?
np.corrcoef(flight_11_transect_small["b"], flight_11_transect_small["s"])

array([[ 1.        , -0.67997827],
       [-0.67997827,  1.        ]])

In [220]:
# Covariance matrix has variance, variance and covariance
np.cov(flight_11_transect_small["h"], flight_11_transect_small["s"]).astype("str")

array([['11741.946726436052', '99.68718835290565'],
       ['99.68718835290565', '1.7953531207024152']], dtype='<U32')

# Look at roughness ratio between surface and bed

In [173]:
fig = go.Figure(data = go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["b"], 
                                  mode = 'markers'))
fig.add_trace(go.Scatter(x = bas_agab[bas_agab["trajectory_id"] == 11]["x"], 
                                  y = bas_agab[bas_agab["trajectory_id"] == 11]["s"], 
                                  mode = 'markers'))
fig.update_layout(title = "Flight line 11", template = "plotly_white")
# Dome A is around 1 M
fig.show()